In [1]:
from collections import defaultdict

In [2]:
occur_thres = 20
ambiguity_thres = 0.97
bos = ['<bos-1>', '<bos-2>']
eos = ['<eos-1>', '<eos-2>']
bop = ['<bop-1>', '<bop-2>']
eop = ['<eop-1>', '<eop-2>']

In [6]:
def get_vocab(train_data):
    word_pos_counter = {}
    pos_vocab = set()
    with open(train_data) as fr:
        for line in fr:
            if not line.strip():
                continue
            word, pos_tag = line.strip().split(' ')
            if word not in word_pos_counter:
                word_pos_counter[word] = {}
            if pos_tag not in word_pos_counter[word]:
                word_pos_counter[word][pos_tag] =0
            word_pos_counter[word][pos_tag] += 1
    
            if pos_tag not in pos_vocab:
                pos_vocab.add(pos_tag)
    
    single_pos_dict = {}
    for word, tag_freqs in word_pos_counter.items():
        tag, mode = max(tag_freqs.items(), key=lambda item: item[1])
        n = sum(tag_freqs.values())
        # Don't add rare words to the tag dictionary
        # Only add quite unambiguous words
        if n >= occur_thres and (float(mode) / n) >= ambiguity_thres:
            single_pos_dict[word] = tag
            
    return single_pos_dict, pos_vocab

In [7]:
single_pos_dict, pos_vocab = get_vocab('./data/wsj/train.pos')

In [10]:
def read_data(path):
    sentence = []
    pos_tag = []
    line_sent = []
    line_pos = []
    for sent in open(path):
        if not sent.strip():

            sentence.append(line_sent)
            pos_tag.append(line_pos)
            
            line_sent = []
            line_pos = []
            continue

        w, p = sent.strip().split(' ')
        line_sent.append(w)
        line_pos.append(p)

    return sentence, pos_tag

In [11]:
train_sent, train_pos = read_data('./data/wsj/train.pos')
print('train size:', len(train_sent))
dev_sent, dev_pos = read_data('./data/wsj/dev.pos')
print('dev size:', len(dev_sent))
test_sent, test_pos = read_data('./data/wsj/test.pos')
print('test size:', len(test_sent))


train size: 38218
dev size: 5526
test size: 5461


In [12]:
def get_features(i, word, context, prev, prev2):
    '''Map tokens into a feature representation, implemented as a
    {hashable: float} dict. If the features change, a new model must be
    trained.
    '''

    def add(name, *args):
        features[' '.join((name,) + tuple(args))] += 1

    i += len(bos)
    features = defaultdict(int)
    # It's useful to have a constant feature, which acts sort of like a prior
    add('bias')
    add('i suffix', word[-3:])
    add('i pref1', word[0])
    add('i-1 tag', prev)
    add('i-2 tag', prev2)
    add('i tag+i-2 tag', prev, prev2)
    add('i word', context[i])
    add('i-1 tag+i word', prev, context[i])
    add('i-1 word', context[i - 1])
    add('i-1 suffix', context[i - 1][-3:])
    add('i-2 word', context[i - 2])
    add('i+1 word', context[i + 1])
    add('i+1 suffix', context[i + 1][-3:])
    add('i+2 word', context[i + 2])
    return features

In [33]:
def process_data(sent_data, pos_data):
    words = []
    feats = []
    trgs = []
    assert len(sent_data) == len(pos_data)
    for sent_seq, pos_seq in zip(sent_data, pos_data):
        sent_seq = bos + sent_seq + eos
        pos_seq = bop + pos_seq + eop
        for idx, (word, pos) in enumerate(zip(sent_seq[2:-2], pos_seq[2:-2])):
            words.append(word)
            trgs.append(pos)
            feats.append(get_features(idx, word, sent_seq, pos_seq[idx-1], pos_seq[idx-2]))
    return words, feats, trgs

In [34]:
train_words, train_feats, train_trgs = process_data(train_sent, train_pos)
dev_words, dev_feats, dev_trgs = process_data(dev_sent, dev_pos)
test_words, test_feats, test_trgs = process_data(test_sent, test_pos)

In [35]:
from perceptron import AveragedPerceptron
import copy
import random
model = AveragedPerceptron()
model.classes = pos_vocab

idx_list = list(range(len(train_words)))
for epoch in range(1, 11):
    num_update = 0
    train_cor = 0
    random.shuffle(idx_list)
    for idx in idx_list:
        word = train_words[idx]
        feats = train_feats[idx]
        trg = train_trgs[idx]
        guess = model.predict(feats)
        if guess != trg:
            num_update += 1
            model.update(trg, guess, feats)
        else:
            train_cor += 1
    model.average_weights()
    model.save("model/normal-{:03d}.pkl".format(epoch))
    
    #valid
    dev_cor = 0
    for idx, (feats, trg) in enumerate(zip(dev_feats, dev_trgs)):
        guess = model.predict(feats)
        if guess == trg:
            dev_cor += 1
    print("Epoch {} | num updates {} ({:.4f}%) | train cor {:.4f}% | dev cor {:.4f}%".format(
        epoch, num_update, num_update/len(train_feats) * 100, train_cor/len(train_feats) * 100, dev_cor/len(dev_feats) * 100))
# model.average_weights()
# model.save('model/normal-weighted.pkl')

Epoch 1 | num updates 70503 (7.7279%) | train cor 92.2721% | dev cor 96.0091%
Epoch 2 | num updates 50926 (5.5820%) | train cor 94.4180% | dev cor 96.2247%
Epoch 3 | num updates 48163 (5.2792%) | train cor 94.7208% | dev cor 96.2998%
Epoch 4 | num updates 47468 (5.2030%) | train cor 94.7970% | dev cor 96.3135%
Epoch 5 | num updates 46644 (5.1127%) | train cor 94.8873% | dev cor 96.3302%
Epoch 6 | num updates 46502 (5.0971%) | train cor 94.9029% | dev cor 96.3590%
Epoch 7 | num updates 46392 (5.0851%) | train cor 94.9149% | dev cor 96.3780%
Epoch 8 | num updates 45812 (5.0215%) | train cor 94.9785% | dev cor 96.3932%
Epoch 9 | num updates 45862 (5.0270%) | train cor 94.9730% | dev cor 96.3902%
Epoch 10 | num updates 45521 (4.9896%) | train cor 95.0104% | dev cor 96.4046%


In [36]:
from perceptron import AveragedPerceptron
import random
model = AveragedPerceptron()
model.classes = pos_vocab

idx_list = list(range(len(train_words)))
for epoch in range(1, 11):
    # train
    num_update = 0
    train_cor = 0
    random.shuffle(idx_list)
    for idx in idx_list:
        word = train_words[idx]
        feats = train_feats[idx]
        trg = train_trgs[idx]
#     for idx, (word, feats, trg) in enumerate(zip(train_words, train_feats, train_trgs)):
        if word in single_pos_dict:
            guess = single_pos_dict[word]
        else:
            guess = model.predict(feats)
            if guess != trg:
                num_update += 1
                model.update(trg, guess, feats)
        if guess == trg:
            train_cor += 1
    model.average_weights()
    model.save("model/plus-{:03d}.pkl".format(epoch))
    
    #valid
    dev_cor = 0
    for idx, (word, feats, trg) in enumerate(zip(dev_words, dev_feats, dev_trgs)):
        if word in single_pos_dict:
            guess = single_pos_dict[word]
        else:
            guess = model.predict(feats)
        if guess == trg:
            dev_cor += 1
    print("Epoch {} | num updates {} ({:4f}%) | train cor {:4f}% | dev cor {:.4f}%".format(
        epoch, num_update, num_update/len(train_feats) * 100, train_cor/len(train_feats) * 100, dev_cor/len(dev_feats) * 100))
# model.average_weights()
# model.save('./model/plus-weighted.pkl')

Epoch 1 | num updates 60006 (6.577312%) | train cor 93.263643% | dev cor 96.0767%
Epoch 2 | num updates 44862 (4.917364%) | train cor 94.923590% | dev cor 96.2475%
Epoch 3 | num updates 42397 (4.647173%) | train cor 95.193781% | dev cor 96.3135%
Epoch 4 | num updates 41547 (4.554004%) | train cor 95.286950% | dev cor 96.3545%
Epoch 5 | num updates 41451 (4.543482%) | train cor 95.297473% | dev cor 96.3568%
Epoch 6 | num updates 40879 (4.480784%) | train cor 95.360170% | dev cor 96.3757%
Epoch 7 | num updates 41142 (4.509612%) | train cor 95.331343% | dev cor 96.3788%
Epoch 8 | num updates 40564 (4.446257%) | train cor 95.394698% | dev cor 96.3871%
Epoch 9 | num updates 40504 (4.439680%) | train cor 95.401275% | dev cor 96.3978%
Epoch 10 | num updates 40339 (4.421594%) | train cor 95.419360% | dev cor 96.4084%


In [38]:
model.load('./model/normal-010.pkl')
test_cor = 0
for idx, (feats, trg) in enumerate(zip(test_feats, test_trgs)):
    guess = model.predict(feats)
    if guess == trg:
        test_cor += 1
print("Normal Test ACC {}%".format(test_cor/len(test_feats)))

model.load('./model/plus-010.pkl')
test_cor = 0
for idx, (word, feats, trg) in enumerate(zip(test_words, test_feats, test_trgs)):
    if word in single_pos_dict:
        guess = single_pos_dict[word]
    else:
        guess = model.predict(feats)
    if guess == trg:
        test_cor += 1
print("Plus Test ACC {}%".format(test_cor/len(test_feats)))

Normal Test ACC 0.9642273035857557%
Plus Test ACC 0.9639418626180337%
